In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
df = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "application_features.parquet")
df.shape

(307511, 142)

## 1. Split Data & Preparation

Split into train/test **before** doing anything else with the target, to avoid any data leakage.
We use `stratify=y` because the target is imbalanced — without it, the test set could randomly
end up with a very different positive rate than the train set, making evaluation unreliable.

In [2]:
from sklearn.model_selection import train_test_split

drop_cols = ["TARGET", "SK_ID_CURR"]
X = df.drop(columns=drop_cols)
y = df["TARGET"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train target rate: {y_train.mean():.4f}, Test target rate: {y_test.mean():.4f}")

Train: (246008, 140), Test: (61503, 140)
Train target rate: 0.0807, Test target rate: 0.0807


## 2. Baseline Model: Logistic Regression

A baseline is not a formality — it's the reference point that tells us how much value the more
complex model (LightGBM) actually adds. If LightGBM barely beats this, that's an important
(and honest) finding to report, not a reason to hide the baseline.

Logistic Regression needs purely numeric input, so categorical columns are excluded here —
this is a limitation of the baseline specifically, not of the final model.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(include=["category"]).columns.tolist()

# Logistic regression butuh data numerik murni -> drop kategorikal untuk baseline
X_train_num = X_train[num_cols]
X_test_num = X_test[num_cols]

baseline_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42))
])

baseline_pipe.fit(X_train_num, y_train)
baseline_proba = baseline_pipe.predict_proba(X_test_num)[:, 1]
baseline_auc = roc_auc_score(y_test, baseline_proba)
print(f"Baseline Logistic Regression AUC: {baseline_auc:.4f}")

Baseline Logistic Regression AUC: 0.7457


## 3. Handling Class Imbalance

Three common approaches exist: class weighting, SMOTE (synthetic oversampling), or undersampling.
**Class weighting is used here as the primary approach** instead of SMOTE, for a specific reason:

SMOTE generates synthetic samples by interpolating between nearest neighbors, which works well
for low-dimensional numeric data — but its benefit is questionable for high-dimensional mixed
numeric/categorical tabular data like this, and it adds pipeline complexity without a guaranteed
improvement. `scale_pos_weight` in LightGBM/XGBoost achieves a similar effect (upweighting the
minority class during training) without the risk of creating unrealistic synthetic records.

In [4]:
pos_rate = y_train.mean()
scale_pos_weight = (1 - pos_rate) / pos_rate
print(f"scale_pos_weight yang disarankan: {scale_pos_weight:.2f}")

scale_pos_weight yang disarankan: 11.39


## 4. LightGBM: Default Parameters, then Optuna-Tuned

Two steps here:
1. Train LightGBM with reasonable default settings first, to see the uplift from switching
   algorithms alone (before any tuning).
2. Tune hyperparameters with Optuna using 3-fold stratified cross-validation, to see the
   additional uplift from tuning specifically.

Keeping these two steps separate makes it possible to answer "how much did the algorithm help"
vs. "how much did tuning help" as two different questions in the README.

### 4a. LightGBM with default parameters

In [5]:
import lightgbm as lgb

for col in cat_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

lgb_train = lgb.Dataset(X_train, label=y_train, categorical_feature=cat_cols)
lgb_test = lgb.Dataset(X_test, label=y_test, categorical_feature=cat_cols, reference=lgb_train)

params_default = {
    "objective": "binary",
    "metric": "auc",
    "scale_pos_weight": scale_pos_weight,
    "verbosity": -1,
    "seed": 42,
}

model_default = lgb.train(
    params_default, lgb_train, num_boost_round=500,
    valid_sets=[lgb_test], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

default_proba = model_default.predict(X_test)
default_auc = roc_auc_score(y_test, default_proba)
print(f"LightGBM default AUC: {default_auc:.4f} (vs baseline {baseline_auc:.4f})")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.769369
[200]	valid_0's auc: 0.77008
Early stopping, best iteration is:
[244]	valid_0's auc: 0.77072
LightGBM default AUC: 0.7707 (vs baseline 0.7457)


### 4b. Hyperparameter tuning with Optuna

Each Optuna "trial" tries one combination of hyperparameters, evaluated with 3-fold
cross-validation (not a single train/test split) so the tuning result is more stable and
less likely to overfit to one particular validation set.

In [6]:
import optuna
from sklearn.model_selection import StratifiedKFold

def objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "scale_pos_weight": scale_pos_weight,
        "verbosity": -1,
        "seed": 42,
        "num_leaves": trial.suggest_int("num_leaves", 20, 150),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.5, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
    }

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    aucs = []
    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        train_set = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols)
        val_set = lgb.Dataset(X_val, label=y_val, categorical_feature=cat_cols, reference=train_set)

        model = lgb.train(
            params, train_set, num_boost_round=300,
            valid_sets=[val_set], callbacks=[lgb.early_stopping(30, verbose=False)]
        )
        proba = model.predict(X_val)
        aucs.append(roc_auc_score(y_val, proba))

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", study_name="credit_risk_lgb")
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"Best CV AUC: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

[I 2026-10-09 21:02:27,064] A new study created in memory with name: credit_risk_lgb


  0%|          | 0/30 [00:00<?, ?it/s]

[I 2026-10-09 21:02:47,678] Trial 0 finished with value: 0.7632348600092058 and parameters: {'num_leaves': 66, 'max_depth': 8, 'learning_rate': 0.03462210323776973, 'feature_fraction': 0.549925186458063, 'bagging_fraction': 0.5338224793945736, 'bagging_freq': 1, 'min_child_samples': 82, 'lambda_l1': 7.986058961390113e-05, 'lambda_l2': 3.1181788222503907e-06}. Best is trial 0 with value: 0.7632348600092058.
[I 2026-10-09 21:03:01,595] Trial 1 finished with value: 0.7618308840209699 and parameters: {'num_leaves': 38, 'max_depth': 3, 'learning_rate': 0.05907331702896391, 'feature_fraction': 0.5190804243517282, 'bagging_fraction': 0.5947518769550408, 'bagging_freq': 5, 'min_child_samples': 63, 'lambda_l1': 2.5179946942125135, 'lambda_l2': 0.13873288908134004}. Best is trial 0 with value: 0.7632348600092058.
[I 2026-10-09 21:03:21,650] Trial 2 finished with value: 0.7626672688683417 and parameters: {'num_leaves': 73, 'max_depth': 6, 'learning_rate': 0.0373716848377404, 'feature_fraction': 0

### 4c. Train the final model using the best parameters found

In [7]:
best_params = study.best_params
best_params.update({
    "objective": "binary", "metric": "auc",
    "scale_pos_weight": scale_pos_weight, "verbosity": -1, "seed": 42
})

model_final = lgb.train(
    best_params, lgb_train, num_boost_round=1000,
    valid_sets=[lgb_test], callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

final_proba = model_final.predict(X_test)
final_auc = roc_auc_score(y_test, final_proba)
print(f"LightGBM tuned AUC: {final_auc:.4f}")

Training until validation scores don't improve for 50 rounds
[100]	valid_0's auc: 0.764192
[200]	valid_0's auc: 0.772136
[300]	valid_0's auc: 0.774956
[400]	valid_0's auc: 0.775611
Early stopping, best iteration is:
[434]	valid_0's auc: 0.776085
LightGBM tuned AUC: 0.7761


## 5. Full Evaluation: AUC, PR-AUC, and KS Statistic

AUC alone is not enough for imbalanced data — it can look deceptively good while the model is
still poor at identifying the minority class. Two additional metrics are used here:

- **PR-AUC** (Precision-Recall AUC): more sensitive to performance on the minority (default) class.
- **KS statistic**: the standard metric in credit scoring, measuring how well the model separates
  good vs. bad borrowers across all score thresholds. A KS above 0.40 is generally considered
  a strong model in the finance industry, 0.30–0.40 is acceptable, below 0.20 is weak.

In [8]:
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve

def ks_statistic(y_true, y_proba):
    fpr, tpr, _ = roc_curve(y_true, y_proba)
    return np.max(tpr - fpr)

models_compare = {
    "Baseline (Logistic Regression)": baseline_auc,
}

pr_auc_default = average_precision_score(y_test, default_proba)
pr_auc_final = average_precision_score(y_test, final_proba)
ks_default = ks_statistic(y_test, default_proba)
ks_final = ks_statistic(y_test, final_proba)

comparison = pd.DataFrame({
    "Model": ["Logistic Regression (baseline)", "LightGBM (default)", "LightGBM (tuned)"],
    "AUC": [baseline_auc, default_auc, final_auc],
    "PR-AUC": [average_precision_score(y_test, baseline_proba), pr_auc_default, pr_auc_final],
    "KS": [ks_statistic(y_test, baseline_proba), ks_default, ks_final],
}).round(4)

comparison

,Model,AUC,PR-AUC,KS
0,Logistic Regression (baseline),0.7457,0.2259,0.3634
1,LightGBM (default),0.7707,0.2643,0.4003
2,LightGBM (tuned),0.7761,0.2673,0.4116


## 6. Save Model & Results

Save the trained model, best hyperparameters, and evaluation outputs so Day 10
(SHAP interpretation and cost-based threshold tuning) can load them directly
without retraining from scratch.

In [9]:
import joblib

MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
model_final.save_model(str(MODELS_DIR / "lgb_model_final.txt"))
joblib.dump(study.best_params, MODELS_DIR / "best_params.pkl")

comparison.to_csv(PROJECT_ROOT / "data" / "processed" / "model_comparison.csv", index=False)

np.save(PROJECT_ROOT / "data" / "processed" / "y_test.npy", y_test.values)
np.save(PROJECT_ROOT / "data" / "processed" / "final_proba.npy", final_proba)